### Data Import

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, accuracy_score

data_path = Path("data/processed/spotify_clean.csv")
if not data_path.exists():
    data_path = Path("../data/processed/spotify_clean.csv")

df = pd.read_csv(data_path)
print("Repartition of the target (is_popular) :")
print(df['is_popular'].value_counts(normalize=True).round(4) * 100)

Repartition of the target (is_popular) :
is_popular
0    74.24
1    25.76
Name: proportion, dtype: float64


In [5]:
groups = df['track_id']
y = df['is_popular']
X = df.drop(columns=['is_popular', 'track_id'])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

print(f"Train shape : {X_train.shape} | Test shape : {X_test.shape}")

Train shape : (91281, 15) | Test shape : (22718, 15)


In [ ]:
num_cols = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
            'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']
cat_cols = ['explicit', 'key', 'mode', 'time_signature', 'track_genre']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

cv = StratifiedGroupKFold(n_splits=5)
scoring = ['accuracy', 'f1', 'roc_auc']

models = {
    "Dummy Classifier": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', model)])
    scores = cross_validate(pipe, X_train, y_train, groups=groups_train, cv=cv, scoring=scoring, n_jobs=-1)
    
    print(f"\n=== {name} (5-Fold CV) ===")
    print(f"Accuracy : {scores['test_accuracy'].mean():.4f} (+/- {scores['test_accuracy'].std():.4f})")
    print(f"F1-Score : {scores['test_f1'].mean():.4f} (+/- {scores['test_f1'].std():.4f})")
    print(f"ROC-AUC  : {scores['test_roc_auc'].mean():.4f} (+/- {scores['test_roc_auc'].std():.4f})")